# Exploración y modelado: California Housing

Este notebook documenta el flujo completo del proyecto solicitado:

1. Comprender y validar el dataset.
2. Separar los datos en entrenamiento y prueba.
3. Entrenar K-Means con 6 clusters usando `Latitude`, `Longitude` y `MedInc`.
4. Visualizar y analizar los clusters.
5. Entrenar un clasificador supervisado con las etiquetas generadas.
6. Evaluar y guardar los modelos.

La implementación productiva se encuentra en `src/app.py`; aquí se muestran sus pasos de forma reproducible y explicada.

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_PATH = ROOT_DIR / "data" / "housing.csv"
FEATURES = ["Latitude", "Longitude", "MedInc"]
RANDOM_STATE = 42
N_CLUSTERS = 6

# Permite importar las funciones de la aplicación si el notebook se ejecuta desde src/.
sys.path.insert(0, str(ROOT_DIR / "src"))

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 1. Carga y comprensión del dataset

```python
housing = pd.read_csv(DATA_PATH)
print(f"Dimensiones: {housing.shape}")
display(housing.head())
display(housing[FEATURES].describe().T)
print("Valores nulos:")
display(housing[FEATURES].isna().sum())
```

El dataset contiene 20.640 registros. Para cumplir el enunciado, el modelado utiliza únicamente las tres variables indicadas.

## 2. División train/test y exploración visual

```python
X = housing[FEATURES].copy()
X_train, X_test = train_test_split(
    X, test_size=0.2, random_state=RANDOM_STATE
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].scatter(X_train["Longitude"], X_train["Latitude"], s=5, alpha=0.25)
axes[0].set_title("Ubicación geográfica: train")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

sns.histplot(X_train["MedInc"], kde=True, ax=axes[1])
axes[1].set_title("Distribución de MedInc: train")
axes[1].set_xlabel("Ingreso medio")
plt.tight_layout()
plt.show()
```

## 3. Entrenamiento de K-Means

Se estandarizan las variables para evitar que sus escalas influyan de forma desproporcionada en las distancias del algoritmo.

```python
kmeans_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(
        n_clusters=N_CLUSTERS,
        random_state=RANDOM_STATE,
        n_init=20,
    )),
])

kmeans_pipeline.fit(X_train)
train_clusters = kmeans_pipeline.predict(X_train)
test_clusters = kmeans_pipeline.predict(X_test)

train_labeled = X_train.copy()
test_labeled = X_test.copy()
train_labeled["cluster"] = train_clusters
test_labeled["cluster"] = test_clusters

print("Clusters encontrados:", sorted(train_labeled["cluster"].unique()))
print("Silhouette score en train:", round(
    silhouette_score(X_train, train_clusters), 4
))
display(train_labeled.groupby("cluster")[FEATURES].mean().round(2))
```

## 4. Visualización de los clusters y predicción en test

```python
fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(
    train_labeled["Longitude"], train_labeled["Latitude"],
    c=train_labeled["cluster"], cmap="tab10", s=8, alpha=0.35,
    label="Train",
)
ax.scatter(
    test_labeled["Longitude"], test_labeled["Latitude"],
    c=test_labeled["cluster"], cmap="tab10", s=20, marker="x",
    alpha=0.8, label="Test predicho",
)
ax.set_title("California Housing: K-Means con 6 clusters")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.tight_layout()
plt.show()
```

La figura permite comprobar la distribución geográfica de los clusters y observar los puntos del conjunto de prueba incorporados mediante `kmeans_pipeline.predict`.

## 5. Modelo supervisado

El modelo supervisado aprende las etiquetas producidas por K-Means en el conjunto de entrenamiento. La evaluación compara sus predicciones con las etiquetas K-Means del conjunto de prueba.

```python
classifier = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)
classifier.fit(X_train, train_clusters)
classifier_test_clusters = classifier.predict(X_test)

accuracy = accuracy_score(test_clusters, classifier_test_clusters)
print(f"Accuracy frente a las etiquetas K-Means: {accuracy:.4f}")
print(classification_report(
    test_clusters, classifier_test_clusters, zero_division=0
))
```

Una exactitud alta significa que el clasificador puede aproximar la frontera de decisión de K-Means para nuevos puntos. No debe interpretarse como exactitud frente a una etiqueta real externa.

## 6. Artefactos guardados

El script `src/app.py` ejecuta este flujo de forma automatizada y guarda los modelos en `models/`, además de los datasets etiquetados, métricas y gráfico en `data/processed/`.

```python
from pathlib import Path

print("Modelo K-Means:", Path("../models/kmeans_pipeline.joblib"))
print("Modelo supervisado:", Path("../models/cluster_classifier.joblib"))
```

In [3]:
# Ejecución reproducible del análisis completo
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.model_selection import train_test_split

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "src"))
from app import build_models, load_dataset, FEATURES, RANDOM_STATE

housing = load_dataset(ROOT_DIR / "data" / "housing.csv")
X_train, X_test = train_test_split(
    housing[FEATURES], test_size=0.2, random_state=RANDOM_STATE
)
kmeans, classifier = build_models()
kmeans.fit(X_train)
train_clusters = kmeans.predict(X_train)
test_clusters = kmeans.predict(X_test)

train_labeled = X_train.copy()
test_labeled = X_test.copy()
train_labeled["cluster"] = train_clusters
test_labeled["cluster"] = test_clusters

classifier.fit(X_train, train_clusters)
classifier_test_clusters = classifier.predict(X_test)
accuracy = accuracy_score(test_clusters, classifier_test_clusters)

print(f"Dataset: {housing.shape[0]:,} registros")
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Silhouette score train: {silhouette_score(X_train, train_clusters):.4f}")
print(f"Accuracy supervisada frente a K-Means: {accuracy:.4f}")
print("Distribución de clusters en train:")
display(train_labeled["cluster"].value_counts().sort_index())
display(train_labeled.groupby("cluster")[FEATURES].mean().round(2))
print(classification_report(test_clusters, classifier_test_clusters, zero_division=0))

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(X_train["Longitude"], X_train["Latitude"], c=train_clusters, cmap="tab10", s=8, alpha=0.35, label="Train")
ax.scatter(X_test["Longitude"], X_test["Latitude"], c=test_clusters, cmap="tab10", s=20, marker="x", alpha=0.8, label="Test predicho")
ax.set(title="California Housing: K-Means con 6 clusters", xlabel="Longitude", ylabel="Latitude")
ax.legend()
plt.tight_layout()
plt.show()

Dataset: 20,640 registros
Train: 16,512 | Test: 4,128
Silhouette score train: 0.3941
Accuracy supervisada frente a K-Means: 0.9954
Distribución de clusters en train:


cluster
0     453
1    5189
2    2244
3    3379
4    1321
5    3926
Name: count, dtype: int64

,Latitude,Longitude,MedInc
cluster,,,
0,34.90,-119.24,10.44
1,33.82,-117.82,2.78
2,37.70,-121.94,5.52
3,33.87,-118.02,5.31
4,36.08,-119.73,2.63
5,38.35,-121.90,2.83


              precision    recall  f1-score   support

           0       1.00      0.99      1.00       112
           1       1.00      1.00      1.00      1375
           2       0.99      1.00      0.99       515
           3       1.00      1.00      1.00       828
           4       0.99      0.99      0.99       337
           5       1.00      0.99      1.00       961

    accuracy                           1.00      4128
   macro avg       0.99      0.99      0.99      4128
weighted avg       1.00      1.00      1.00      4128

